# EPIC — EXP033 Strength-Matched TP Study

Цель: проверить, действительно ли `LOW_TP` и `HIGH_TP` отличаются по DNA,
или предыдущий результат объяснялся тем, что `LOW_TP` в среднем слабее по csRNA-seq signal.

## Дизайн

Анализируем три strata:

```text
read_sum = 1
read_sum = 2
read_sum = 3–4
```

Внутри каждого stratum формируем **строго сбалансированные HIGH-score и LOW-score TP**.

Баланс делается внутри одинаковых:

```text
contig
exact read_sum
tss_count
read_max
```

То есть HIGH и LOW группы имеют одинаковое распределение:

- по holdout contig;
- по суммарному количеству reads;
- по числу TSS positions внутри 32 bp;
- по максимальному count в одной позиции.

После этого сравниваем:

1. GC / entropy;
2. положение TSS внутри 32-bp bin;
3. bag-of-4mer;
4. position-aware 256-bp one-hot;
5. composition-only probe;
6. 6-mer differences;
7. positional nucleotide-profile differences.

Основной probe считается двумя способами:

```text
random stratified split
contig-held-out GroupKFold
```

В конце **все результаты автоматически собираются в один JSON-файл**:

```text
EXP033_STRENGTH_MATCHED_TP_RESULTS.json
```

Именно этот файл можно потом загрузить в ChatGPT целиком.


In [1]:
from pathlib import Path
import copy
import gc
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch_directml

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (
        CURRENT_DIR,
        *CURRENT_DIR.parents,
    )
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

from ml_project.epic_baseline import (
    load_baseline_sequences,
)

from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
)

from ml_project.epic_data import (
    load_split,
)

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA / WINDOW CONFIG
# ============================================================

split = load_split(
    PROJECT_ROOT
)

sequences, fasta_alphabet = (
    load_baseline_sequences(
        split
    )
)

train_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "train"
            )[
                "contig"
            ].astype(str)
        )
    )
)

validation_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "validation"
            )[
                "contig"
            ].astype(str)
        )
    )
)

assert set(
    train_contigs
).isdisjoint(
    validation_contigs
)

# Keep 2048-bp target so the regional grid is directly comparable
# with the recent regional experiments.
TARGET_LENGTH = 2048
CONTEXT_FLANK = 130

REGION_WIDTHS = (
    32,
)

PRIMARY_WIDTH = 32

BLOCK_SETTINGS_RF2053 = (
    (7, 1),
    (3, 2),
    (3, 4),
    (3, 8),
    (3, 16),
    (3, 32),
    (3, 64),
    (3, 128),
    (3, 256),
)

BACKBONE_RF_BP = (
    1
    + sum(
        2 * (kernel_size - 1) * dilation
        for kernel_size, dilation
        in BLOCK_SETTINGS_RF2053
    )
)

REGIONAL_UNION_RF_BP = (
    BACKBONE_RF_BP
    + PRIMARY_WIDTH
    - 1
)

assert BACKBONE_RF_BP == 2053
assert REGIONAL_UNION_RF_BP == 2084

assert TARGET_LENGTH % 32 == 0

profile_config = (
    ProfileWindowConfig(
        target_length=TARGET_LENGTH,
        context_flank=CONTEXT_FLANK,
        positive_branch_fraction=0.5,
    )
)

train_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=42,
    )
)

validation_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="validation",
        config=profile_config,
        seed=42,
    )
)

display(split.summary)

print("Train contigs:", len(train_contigs))
print(
    "Validation contigs:",
    len(validation_contigs),
)
print(
    "Input length:",
    TARGET_LENGTH
    + 2 * CONTEXT_FLANK,
)
print(
    "Target length:",
    TARGET_LENGTH,
)
print(
    "Region widths:",
    REGION_WIDTHS,
)

print(
    "Backbone RF:",
    BACKBONE_RF_BP,
    "bp",
)

print(
    "32-bp regional union RF:",
    REGIONAL_UNION_RF_BP,
    "bp",
)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Input length: 2308
Target length: 2048
Region widths: (32,)
Backbone RF: 2053 bp
32-bp regional union RF: 2084 bp


In [4]:
# ============================================================
# FULL HOLDOUT TILE INDEX
# ============================================================

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print(
    "Tiles:",
    len(
        validation_tiles
    ),
)

print(
    "Position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)

print(
    "Positives:",
    int(
        validation_tiles[
            "positive_positions"
        ].sum()
    ),
)

print(
    "Contigs:",
    validation_tiles[
        "contig"
    ].nunique(),
)


Tiles: 24700
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [5]:
# ============================================================
# LOAD POSITIVE-REGION CATALOG FROM THE PREVIOUS LOW_TP STUDY
# ============================================================

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP033_BASELEVEL_RF2053_REGIONAL_32_ONLY"
    / "run_001"
)

PREVIOUS_STUDY_DIR = (
    RUN_DIR
    / "low_tp_study"
)

CATALOG_PATH = (
    PREVIOUS_STUDY_DIR
    / "positive_region_catalog.csv"
)

if not CATALOG_PATH.is_file():
    raise FileNotFoundError(
        "Не найден positive_region_catalog.csv.\n"
        "Сначала запусти последнюю SAVE-ячейку "
        "в EPIC_EXP033_LOW_TP_STUDY.ipynb.\n"
        f"Ожидался файл: {CATALOG_PATH}"
    )

positive_regions = pd.read_csv(
    CATALOG_PATH
)

required_columns = {
    "catalog_index",
    "score",
    "selected_top10",
    "tss_count",
    "read_sum",
    "read_max",
    "closest_tss_abs_offset_to_region_center",
    "dominant_tss_signed_offset_from_center",
    "dominant_tss_distance_to_nearest_edge",
    "tile_row",
    "strand_index",
    "region_index",
}

missing_columns = (
    required_columns
    - set(
        positive_regions.columns
    )
)

if missing_columns:
    raise ValueError(
        "В positive_region_catalog.csv не хватает колонок: "
        + ", ".join(
            sorted(
                missing_columns
            )
        )
    )

# Counts are integer-valued biologically; keep an explicit integer copy.
if not np.allclose(
    positive_regions["read_sum"],
    np.rint(
        positive_regions["read_sum"]
    ),
):
    raise ValueError(
        "read_sum содержит нецелые значения — проверь источник."
    )

if not np.allclose(
    positive_regions["read_max"],
    np.rint(
        positive_regions["read_max"]
    ),
):
    raise ValueError(
        "read_max содержит нецелые значения — проверь источник."
    )

positive_regions[
    "read_sum_int"
] = np.rint(
    positive_regions[
        "read_sum"
    ]
).astype(
    int
)

positive_regions[
    "read_max_int"
] = np.rint(
    positive_regions[
        "read_max"
    ]
).astype(
    int
)

positive_regions[
    "tss_count_int"
] = positive_regions[
    "tss_count"
].astype(
    int
)

# Restore holdout contig from the tile index.
tile_to_contig = (
    validation_tiles[
        "contig"
    ]
    .astype(
        str
    )
    .to_dict()
)

positive_regions[
    "contig"
] = positive_regions[
    "tile_row"
].map(
    tile_to_contig
)

if positive_regions[
    "contig"
].isna().any():
    raise RuntimeError(
        "Не удалось восстановить contig для части positive regions."
    )

print(
    "Positive regions loaded:",
    f"{len(positive_regions):,}",
)

print(
    "Holdout contigs:",
    positive_regions[
        "contig"
    ].nunique(),
)

display(
    positive_regions[
        [
            "score",
            "read_sum_int",
            "read_max_int",
            "tss_count_int",
            "contig",
        ]
    ]
    .head()
)


Positive regions loaded: 27,982
Holdout contigs: 3


,score,read_sum_int,read_max_int,tss_count_int,contig
0,0.093229,223,222,2,NC_064034.1
1,0.258139,1,1,1,NC_064034.1
2,0.302300,7,7,1,NC_064034.1
3,0.982676,1,1,1,NC_064034.1
4,0.960754,1,1,1,NC_064034.1


In [6]:
# ============================================================
# BUILD STRICT STRENGTH-MATCHED HIGH/LOW SCORE COHORTS
# ============================================================

TAIL_FRACTION = 0.25
MIN_CELL_SIZE = 8

STRATA = {
    "READS_1": (
        1,
    ),
    "READS_2": (
        2,
    ),
    "READS_3_4": (
        3,
        4,
    ),
}

MATCH_COLUMNS = [
    "contig",
    "read_sum_int",
    "tss_count_int",
    "read_max_int",
]


def build_strict_score_tails(
    frame,
    stratum_name,
    allowed_read_sums,
):
    eligible = (
        frame[
            frame[
                "read_sum_int"
            ].isin(
                allowed_read_sums
            )
        ]
        .copy()
    )

    selected_parts = []
    cell_rows = []

    for key, group in eligible.groupby(
        MATCH_COLUMNS,
        sort=False,
    ):
        group = group.sort_values(
            "score",
            kind="mergesort",
        )

        n = len(
            group
        )

        if n < MIN_CELL_SIZE:
            continue

        n_tail = int(
            np.floor(
                n
                * TAIL_FRACTION
            )
        )

        if n_tail < 1:
            continue

        # TAIL_FRACTION=0.25 guarantees no overlap.
        low = (
            group.iloc[
                :n_tail
            ]
            .copy()
        )

        high = (
            group.iloc[
                -n_tail:
            ]
            .copy()
        )

        low[
            "score_group"
        ] = "LOW_SCORE_TP"

        high[
            "score_group"
        ] = "HIGH_SCORE_TP"

        low[
            "stratum"
        ] = stratum_name

        high[
            "stratum"
        ] = stratum_name

        low[
            "match_cell"
        ] = str(
            key
        )

        high[
            "match_cell"
        ] = str(
            key
        )

        selected_parts.extend(
            [
                low,
                high,
            ]
        )

        cell_rows.append({
            "stratum": stratum_name,
            "contig": key[0],
            "read_sum_int": int(
                key[1]
            ),
            "tss_count_int": int(
                key[2]
            ),
            "read_max_int": int(
                key[3]
            ),
            "cell_n": int(
                n
            ),
            "selected_per_tail": int(
                n_tail
            ),
        })

    if not selected_parts:
        raise RuntimeError(
            f"Не удалось сформировать cohort для {stratum_name}."
        )

    selected = pd.concat(
        selected_parts,
        ignore_index=True,
    )

    cells_df = pd.DataFrame(
        cell_rows
    )

    return (
        eligible,
        selected,
        cells_df,
    )


cohort_parts = []
match_cell_parts = []
retention_rows = []

for stratum_name, allowed_read_sums in STRATA.items():
    eligible, selected, match_cells = (
        build_strict_score_tails(
            positive_regions,
            stratum_name,
            allowed_read_sums,
        )
    )

    cohort_parts.append(
        selected
    )

    match_cell_parts.append(
        match_cells
    )

    n_each = (
        selected[
            "score_group"
        ]
        .value_counts()
    )

    retention_rows.append({
        "stratum": stratum_name,
        "eligible_positive_regions": int(
            len(
                eligible
            )
        ),
        "matched_low_n": int(
            n_each.get(
                "LOW_SCORE_TP",
                0,
            )
        ),
        "matched_high_n": int(
            n_each.get(
                "HIGH_SCORE_TP",
                0,
            )
        ),
        "matched_fraction_of_eligible_per_tail": float(
            n_each.get(
                "LOW_SCORE_TP",
                0,
            )
            / len(
                eligible
            )
        ),
        "match_cells_used": int(
            len(
                match_cells
            )
        ),
    })

matched = pd.concat(
    cohort_parts,
    ignore_index=True,
)

match_cells = pd.concat(
    match_cell_parts,
    ignore_index=True,
)

retention_summary = pd.DataFrame(
    retention_rows
)

display(
    retention_summary
)

# Exact balance check:
balance = (
    matched
    .groupby(
        [
            "stratum",
            *MATCH_COLUMNS,
            "score_group",
        ],
        as_index=False,
    )
    .size()
)

balance_pivot = (
    balance
    .pivot_table(
        index=[
            "stratum",
            *MATCH_COLUMNS,
        ],
        columns="score_group",
        values="size",
        fill_value=0,
    )
    .reset_index()
)

balance_pivot[
    "abs_n_difference"
] = np.abs(
    balance_pivot.get(
        "HIGH_SCORE_TP",
        0,
    )
    - balance_pivot.get(
        "LOW_SCORE_TP",
        0,
    )
)

if (
    balance_pivot[
        "abs_n_difference"
    ].max()
    != 0
):
    raise RuntimeError(
        "Strict matching failed: HIGH/LOW counts differ inside a match cell."
    )

print(
    "Strict matching balance: OK"
)

print(
    "Total selected:",
    f"{len(matched):,}",
)

display(
    matched
    .groupby(
        [
            "stratum",
            "score_group",
        ],
        as_index=False,
    )
    .agg(
        n=(
            "score",
            "size",
        ),
        score_median=(
            "score",
            "median",
        ),
        read_sum_median=(
            "read_sum_int",
            "median",
        ),
        read_max_median=(
            "read_max_int",
            "median",
        ),
        tss_count_median=(
            "tss_count_int",
            "median",
        ),
    )
)


,stratum,eligible_positive_regions,matched_low_n,matched_high_n,matched_fraction_of_eligible_per_tail,match_cells_used
0,READS_1,13611,3402,3402,0.249945,3
1,READS_2,5194,1296,1296,0.249519,6
2,READS_3_4,3520,872,872,0.247727,24


Strict matching balance: OK
Total selected: 11,140


,stratum,score_group,n,score_median,read_sum_median,read_max_median,tss_count_median
0,READS_1,HIGH_SCORE_TP,3402,0.942000,1.0,1.0,1.0
1,READS_1,LOW_SCORE_TP,3402,0.292174,1.0,1.0,1.0
2,READS_2,HIGH_SCORE_TP,1296,0.956032,2.0,2.0,1.0
3,READS_2,LOW_SCORE_TP,1296,0.361418,2.0,2.0,1.0
4,READS_3_4,HIGH_SCORE_TP,872,0.968472,3.0,2.0,2.0
5,READS_3_4,LOW_SCORE_TP,872,0.431429,3.0,2.0,2.0


## Почему matching строгий

Особенно важно для `READS_3_4`.

Мы **не просто** смешиваем все `read_sum=3` и `read_sum=4`.

HIGH и LOW выбираются внутри каждого одинакового набора:

```text
contig + exact read_sum + tss_count + read_max
```

Поэтому различия DNA уже нельзя просто объяснить тем, что одна группа содержит больше
`read_sum=4`, больше TSS positions или другой holdout contig.


In [7]:
# ============================================================
# EXTRACT 256-BP STRAND-ORIENTED DNA
# ============================================================

CONTEXT_BP = 256
HALF_CONTEXT = (
    CONTEXT_BP
    // 2
)

REGION_WIDTH = 32
N_REGIONS = (
    TARGET_LENGTH
    // REGION_WIDTH
)

BASES = np.array(
    list(
        "ACGTN"
    )
)

matched[
    "seq_core32"
] = ""

matched[
    "seq_context256"
] = ""

tile_groups = (
    matched
    .groupby(
        "tile_row"
    )
    .indices
)

unique_tile_rows = np.array(
    sorted(
        tile_groups.keys()
    ),
    dtype=np.int64,
)

EXTRACT_BATCH_SIZE = 64

for batch_start in range(
    0,
    len(
        unique_tile_rows
    ),
    EXTRACT_BATCH_SIZE,
):
    tile_rows = (
        unique_tile_rows[
            batch_start:
            batch_start
            + EXTRACT_BATCH_SIZE
        ]
    )

    batch = (
        validation_generator
        .make_batch(
            validation_tiles.iloc[
                tile_rows
            ]
        )
    )

    x = (
        batch
        .x_both_strands
        .numpy()
    )

    B = len(
        tile_rows
    )

    tile_to_local = {
        int(tile_row): local
        for local, tile_row
        in enumerate(
            tile_rows
        )
    }

    for tile_row in tile_rows:
        local = (
            tile_to_local[
                int(
                    tile_row
                )
            ]
        )

        for row_idx in (
            tile_groups[
                int(
                    tile_row
                )
            ]
        ):
            strand_index = int(
                matched.at[
                    row_idx,
                    "strand_index",
                ]
            )

            genomic_region = int(
                matched.at[
                    row_idx,
                    "region_index",
                ]
            )

            if strand_index == 0:
                oriented_row = (
                    local
                )

                oriented_region = (
                    genomic_region
                )
            else:
                oriented_row = (
                    B + local
                )

                oriented_region = (
                    N_REGIONS
                    - 1
                    - genomic_region
                )

            core_start = (
                CONTEXT_FLANK
                + oriented_region
                * REGION_WIDTH
            )

            core_end = (
                core_start
                + REGION_WIDTH
            )

            center = (
                core_start
                + REGION_WIDTH // 2
            )

            context_start = (
                center
                - HALF_CONTEXT
            )

            context_end = (
                center
                + HALF_CONTEXT
            )

            if (
                context_start < 0
                or context_end > x.shape[-1]
            ):
                raise RuntimeError(
                    "256-bp context вышел за границы input window."
                )

            core_codes = np.argmax(
                x[
                    oriented_row,
                    :,
                    core_start:
                    core_end,
                ],
                axis=0,
            )

            context_codes = np.argmax(
                x[
                    oriented_row,
                    :,
                    context_start:
                    context_end,
                ],
                axis=0,
            )

            matched.at[
                row_idx,
                "seq_core32",
            ] = "".join(
                BASES[
                    core_codes
                ]
            )

            matched.at[
                row_idx,
                "seq_context256",
            ] = "".join(
                BASES[
                    context_codes
                ]
            )

assert (
    matched[
        "seq_context256"
    ]
    .str.len()
    .eq(
        CONTEXT_BP
    )
    .all()
)

assert (
    matched[
        "seq_core32"
    ]
    .str.len()
    .eq(
        REGION_WIDTH
    )
    .all()
)

print(
    "Extracted:",
    f"{len(matched):,}",
    "strand-oriented sequences",
)


Extracted: 11,140 strand-oriented sequences


In [8]:
# ============================================================
# COMPOSITION + LOCALIZATION SUMMARY
# ============================================================

def sequence_stats(
    seq,
):
    counts = {
        b: seq.count(
            b
        )
        for b in "ACGTN"
    }

    valid = sum(
        counts[
            b
        ]
        for b in "ACGT"
    )

    if valid > 0:
        gc = (
            counts[
                "G"
            ]
            + counts[
                "C"
            ]
        ) / valid
    else:
        gc = np.nan

    probs = np.array(
        [
            counts[
                b
            ]
            for b in "ACGT"
        ],
        dtype=np.float64,
    )

    if probs.sum() > 0:
        probs /= probs.sum()

        nz = (
            probs > 0
        )

        entropy = float(
            -np.sum(
                probs[
                    nz
                ]
                * np.log2(
                    probs[
                        nz
                    ]
                )
            )
        )
    else:
        entropy = np.nan

    n_fraction = (
        counts[
            "N"
        ]
        / len(
            seq
        )
    )

    return (
        gc,
        n_fraction,
        entropy,
    )


core_stats = np.array(
    [
        sequence_stats(
            seq
        )
        for seq in (
            matched[
                "seq_core32"
            ]
        )
    ]
)

context_stats = np.array(
    [
        sequence_stats(
            seq
        )
        for seq in (
            matched[
                "seq_context256"
            ]
        )
    ]
)

matched[
    "gc_core32"
] = core_stats[
    :,
    0,
]

matched[
    "n_core32"
] = core_stats[
    :,
    1,
]

matched[
    "entropy_core32"
] = core_stats[
    :,
    2,
]

matched[
    "gc_context256"
] = context_stats[
    :,
    0,
]

matched[
    "n_context256"
] = context_stats[
    :,
    1,
]

matched[
    "entropy_context256"
] = context_stats[
    :,
    2,
]

cohort_summary = (
    matched
    .groupby(
        [
            "stratum",
            "score_group",
        ],
        as_index=False,
    )
    .agg(
        n=(
            "score",
            "size",
        ),
        score_median=(
            "score",
            "median",
        ),
        score_mean=(
            "score",
            "mean",
        ),
        read_sum_mean=(
            "read_sum_int",
            "mean",
        ),
        tss_count_mean=(
            "tss_count_int",
            "mean",
        ),
        read_max_mean=(
            "read_max_int",
            "mean",
        ),
        gc_core32_mean=(
            "gc_core32",
            "mean",
        ),
        gc_context256_mean=(
            "gc_context256",
            "mean",
        ),
        entropy_core32_mean=(
            "entropy_core32",
            "mean",
        ),
        entropy_context256_mean=(
            "entropy_context256",
            "mean",
        ),
        closest_tss_to_center_median_bp=(
            "closest_tss_abs_offset_to_region_center",
            "median",
        ),
        dominant_tss_edge_distance_median_bp=(
            "dominant_tss_distance_to_nearest_edge",
            "median",
        ),
        dominant_tss_signed_offset_median_bp=(
            "dominant_tss_signed_offset_from_center",
            "median",
        ),
    )
)

display(
    cohort_summary
)


,stratum,score_group,n,score_median,score_mean,read_sum_mean,tss_count_mean,read_max_mean,gc_core32_mean,gc_context256_mean,entropy_core32_mean,entropy_context256_mean,closest_tss_to_center_median_bp,dominant_tss_edge_distance_median_bp,dominant_tss_signed_offset_median_bp
0,READS_1,HIGH_SCORE_TP,3402,0.942000,0.941827,1.000000,1.000000,1.000000,0.464093,0.438986,1.904155,1.966922,7.5,8.0,-0.5
1,READS_1,LOW_SCORE_TP,3402,0.292174,0.274626,1.000000,1.000000,1.000000,0.411532,0.403111,1.876088,1.948421,8.5,7.0,-0.5
2,READS_2,HIGH_SCORE_TP,1296,0.956032,0.953770,2.000000,1.308642,1.691358,0.459129,0.437361,1.904870,1.968635,6.5,7.0,-1.5
3,READS_2,LOW_SCORE_TP,1296,0.361418,0.337991,2.000000,1.308642,1.691358,0.406829,0.404318,1.882106,1.948384,7.5,7.0,-2.5
4,READS_3_4,HIGH_SCORE_TP,872,0.968472,0.965204,3.353211,1.815367,2.494266,0.461690,0.436367,1.908642,1.968504,5.5,7.0,-0.5
5,READS_3_4,LOW_SCORE_TP,872,0.431429,0.412274,3.353211,1.815367,2.494266,0.417037,0.408566,1.885050,1.953659,5.5,7.0,-1.5


In [9]:
# ============================================================
# FEATURE BUILDERS + RANDOM / CONTIG-HELD-OUT PROBES
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    GroupKFold,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

BASE_TO_INT = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
}


def kmer_vector(
    seq,
    k=4,
):
    vector = np.zeros(
        4 ** k,
        dtype=np.float32,
    )

    for i in range(
        len(seq)
        - k
        + 1
    ):
        value = 0
        valid = True

        for char in seq[
            i:
            i + k
        ]:
            code = (
                BASE_TO_INT
                .get(
                    char
                )
            )

            if code is None:
                valid = False
                break

            value = (
                value * 4
                + code
            )

        if valid:
            vector[
                value
            ] += 1.0

    total = (
        vector.sum()
    )

    if total > 0:
        vector /= total

    return vector


def position_onehot(
    seq,
):
    vector = np.zeros(
        CONTEXT_BP
        * 4,
        dtype=np.float32,
    )

    for pos, char in enumerate(
        seq
    ):
        code = (
            BASE_TO_INT
            .get(
                char
            )
        )

        if code is not None:
            vector[
                pos * 4
                + code
            ] = 1.0

    return vector


def make_classifier(
    scale=False,
):
    clf = LogisticRegression(
        max_iter=2000,
        C=1.0,
        solver="liblinear",
        random_state=42,
    )

    if scale:
        return make_pipeline(
            StandardScaler(),
            clf,
        )

    return clf


def random_split_auc(
    X,
    y,
    scale=False,
):
    idx = np.arange(
        len(
            y
        )
    )

    train_idx, test_idx = train_test_split(
        idx,
        test_size=0.30,
        random_state=42,
        stratify=y,
    )

    clf = make_classifier(
        scale=scale
    )

    clf.fit(
        X[
            train_idx
        ],
        y[
            train_idx
        ],
    )

    prob = (
        clf
        .predict_proba(
            X[
                test_idx
            ]
        )[
            :,
            1
        ]
    )

    return float(
        roc_auc_score(
            y[
                test_idx
            ],
            prob,
        )
    )


def contig_heldout_auc(
    X,
    y,
    groups,
    scale=False,
):
    unique_groups = np.unique(
        groups
    )

    if len(
        unique_groups
    ) < 2:
        return np.nan

    splitter = GroupKFold(
        n_splits=len(
            unique_groups
        )
    )

    oof_y = []
    oof_prob = []
    fold_rows = []

    for fold_no, (
        train_idx,
        test_idx,
    ) in enumerate(
        splitter.split(
            X,
            y,
            groups,
        ),
        1,
    ):
        y_test = (
            y[
                test_idx
            ]
        )

        if np.unique(
            y_test
        ).size < 2:
            continue

        clf = make_classifier(
            scale=scale
        )

        clf.fit(
            X[
                train_idx
            ],
            y[
                train_idx
            ],
        )

        prob = (
            clf
            .predict_proba(
                X[
                    test_idx
                ]
            )[
                :,
                1
            ]
        )

        fold_auc = float(
            roc_auc_score(
                y_test,
                prob,
            )
        )

        fold_rows.append({
            "fold": int(
                fold_no
            ),
            "heldout_contig": str(
                np.unique(
                    groups[
                        test_idx
                    ]
                )[
                    0
                ]
            ),
            "n_test": int(
                len(
                    test_idx
                )
            ),
            "roc_auc": fold_auc,
        })

        oof_y.append(
            y_test
        )

        oof_prob.append(
            prob
        )

    if not oof_y:
        return (
            np.nan,
            pd.DataFrame(),
        )

    pooled_auc = float(
        roc_auc_score(
            np.concatenate(
                oof_y
            ),
            np.concatenate(
                oof_prob
            ),
        )
    )

    return (
        pooled_auc,
        pd.DataFrame(
            fold_rows
        ),
    )


probe_rows = []
fold_rows_all = []

for stratum_name in STRATA:
    subset = (
        matched[
            matched[
                "stratum"
            ]
            == stratum_name
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    y = (
        subset[
            "score_group"
        ]
        .eq(
            "LOW_SCORE_TP"
        )
        .astype(
            np.int8
        )
        .to_numpy()
    )

    groups = (
        subset[
            "contig"
        ]
        .astype(
            str
        )
        .to_numpy()
    )

    X_composition = (
        subset[
            [
                "gc_core32",
                "gc_context256",
                "entropy_core32",
                "entropy_context256",
            ]
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    X_4mer = np.stack(
        [
            kmer_vector(
                seq,
                k=4,
            )
            for seq in (
                subset[
                    "seq_context256"
                ]
            )
        ]
    )

    X_position = np.stack(
        [
            position_onehot(
                seq
            )
            for seq in (
                subset[
                    "seq_context256"
                ]
            )
        ]
    )

    feature_sets = {
        "composition_4_features": (
            X_composition,
            True,
        ),
        "bag_of_4mer_256bp": (
            X_4mer,
            False,
        ),
        "position_aware_base_256bp": (
            X_position,
            False,
        ),
    }

    for feature_name, (
        X,
        scale,
    ) in feature_sets.items():

        random_auc = (
            random_split_auc(
                X,
                y,
                scale=scale,
            )
        )

        contig_auc, fold_table = (
            contig_heldout_auc(
                X,
                y,
                groups,
                scale=scale,
            )
        )

        probe_rows.append({
            "stratum": stratum_name,
            "feature_set": feature_name,
            "n_total": int(
                len(
                    subset
                )
            ),
            "random_split_auc": float(
                random_auc
            ),
            "contig_heldout_auc": float(
                contig_auc
            ),
        })

        if not fold_table.empty:
            fold_table[
                "stratum"
            ] = stratum_name

            fold_table[
                "feature_set"
            ] = feature_name

            fold_rows_all.append(
                fold_table
            )

probe_results = pd.DataFrame(
    probe_rows
)

probe_folds = (
    pd.concat(
        fold_rows_all,
        ignore_index=True,
    )
    if fold_rows_all
    else pd.DataFrame()
)

display(
    probe_results
)

if not probe_folds.empty:
    display(
        probe_folds
    )


,stratum,feature_set,n_total,random_split_auc,contig_heldout_auc
0,READS_1,composition_4_features,6804,0.684818,0.682043
1,READS_1,bag_of_4mer_256bp,6804,0.756455,0.748976
2,READS_1,position_aware_base_256bp,6804,0.712369,0.697598
3,READS_2,composition_4_features,2592,0.688768,0.688409
4,READS_2,bag_of_4mer_256bp,2592,0.726125,0.729741
5,READS_2,position_aware_base_256bp,2592,0.662017,0.643870
6,READS_3_4,composition_4_features,1744,0.643392,0.650932
7,READS_3_4,bag_of_4mer_256bp,1744,0.687008,0.696749
8,READS_3_4,position_aware_base_256bp,1744,0.580968,0.583788


,fold,heldout_contig,n_test,roc_auc,stratum,feature_set
0,1,NC_064034.1,2846,0.678520,READS_1,composition_4_features
1,2,NC_064042.1,2216,0.691497,READS_1,composition_4_features
2,3,NC_064041.1,1742,0.675725,READS_1,composition_4_features
3,1,NC_064034.1,2846,0.728349,READS_1,bag_of_4mer_256bp
4,2,NC_064042.1,2216,0.769417,READS_1,bag_of_4mer_256bp
5,3,NC_064041.1,1742,0.756453,READS_1,bag_of_4mer_256bp
6,1,NC_064034.1,2846,0.698583,READS_1,position_aware_base_256bp
7,2,NC_064042.1,2216,0.701959,READS_1,position_aware_base_256bp
8,3,NC_064041.1,1742,0.691128,READS_1,position_aware_base_256bp
9,1,NC_064034.1,1082,0.693354,READS_2,composition_4_features


In [10]:
# ============================================================
# POSITIONAL NUCLEOTIDE PROFILE DIFFERENCES
# ============================================================

position_profile_rows = []
position_profile_full_rows = []

for stratum_name in STRATA:
    subset = (
        matched[
            matched[
                "stratum"
            ]
            == stratum_name
        ]
    )

    group_profiles = {}

    for score_group in (
        "HIGH_SCORE_TP",
        "LOW_SCORE_TP",
    ):
        seqs = (
            subset.loc[
                subset[
                    "score_group"
                ]
                == score_group,
                "seq_context256",
            ]
            .tolist()
        )

        onehot = np.stack(
            [
                position_onehot(
                    seq
                )
                .reshape(
                    CONTEXT_BP,
                    4,
                )
                for seq in seqs
            ]
        )

        profile = (
            onehot.mean(
                axis=0
            )
        )

        group_profiles[
            score_group
        ] = profile

        for pos in range(
            CONTEXT_BP
        ):
            for base_index, base in enumerate(
                "ACGT"
            ):
                position_profile_full_rows.append({
                    "stratum": stratum_name,
                    "score_group": score_group,
                    "position_0based": int(
                        pos
                    ),
                    "relative_position_from_center": float(
                        pos
                        - (
                            CONTEXT_BP
                            / 2.0
                            - 0.5
                        )
                    ),
                    "base": base,
                    "frequency": float(
                        profile[
                            pos,
                            base_index,
                        ]
                    ),
                })

    diff = np.abs(
        group_profiles[
            "LOW_SCORE_TP"
        ]
        - group_profiles[
            "HIGH_SCORE_TP"
        ]
    )

    core_start = (
        CONTEXT_BP
        // 2
        - REGION_WIDTH
        // 2
    )

    core_end = (
        core_start
        + REGION_WIDTH
    )

    core_diff = (
        diff[
            core_start:
            core_end
        ]
    )

    position_profile_rows.append({
        "stratum": stratum_name,
        "max_abs_base_frequency_diff_256bp": float(
            diff.max()
        ),
        "mean_abs_base_frequency_diff_256bp": float(
            diff.mean()
        ),
        "max_abs_base_frequency_diff_core32": float(
            core_diff.max()
        ),
        "mean_abs_base_frequency_diff_core32": float(
            core_diff.mean()
        ),
    })

position_profile_summary = pd.DataFrame(
    position_profile_rows
)

position_profile_full = pd.DataFrame(
    position_profile_full_rows
)

display(
    position_profile_summary
)


,stratum,max_abs_base_frequency_diff_256bp,mean_abs_base_frequency_diff_256bp,max_abs_base_frequency_diff_core32,mean_abs_base_frequency_diff_core32
0,READS_1,0.091417,0.020144,0.073192,0.028228
1,READS_2,0.076389,0.022268,0.075617,0.030273
2,READS_3_4,0.086009,0.022206,0.074541,0.025211


In [11]:
# ============================================================
# 6-MER DIFFERENCES WITHIN EACH STRICTLY MATCHED STRATUM
# ============================================================

from collections import Counter

K = 6
PSEUDOCOUNT = 1.0
VOCAB = (
    4 ** K
)


def aggregate_kmers(
    sequences,
    k,
):
    counter = Counter()
    total = 0

    for seq in sequences:
        for i in range(
            len(
                seq
            )
            - k
            + 1
        ):
            kmer = (
                seq[
                    i:
                    i + k
                ]
            )

            if "N" in kmer:
                continue

            counter[
                kmer
            ] += 1

            total += 1

    return (
        counter,
        total,
    )


kmer_tables = []
top_kmer_tables = []

for stratum_name in STRATA:
    subset = (
        matched[
            matched[
                "stratum"
            ]
            == stratum_name
        ]
    )

    high_sequences = (
        subset.loc[
            subset[
                "score_group"
            ]
            == "HIGH_SCORE_TP",
            "seq_context256",
        ]
        .tolist()
    )

    low_sequences = (
        subset.loc[
            subset[
                "score_group"
            ]
            == "LOW_SCORE_TP",
            "seq_context256",
        ]
        .tolist()
    )

    high_counter, high_total = (
        aggregate_kmers(
            high_sequences,
            K,
        )
    )

    low_counter, low_total = (
        aggregate_kmers(
            low_sequences,
            K,
        )
    )

    all_kmers = sorted(
        set(
            high_counter
        )
        | set(
            low_counter
        )
    )

    rows = []

    for kmer in all_kmers:
        high_freq = (
            high_counter[
                kmer
            ]
            + PSEUDOCOUNT
        ) / (
            high_total
            + PSEUDOCOUNT
            * VOCAB
        )

        low_freq = (
            low_counter[
                kmer
            ]
            + PSEUDOCOUNT
        ) / (
            low_total
            + PSEUDOCOUNT
            * VOCAB
        )

        rows.append({
            "stratum": stratum_name,
            "kmer": kmer,
            "frequency_LOW": float(
                low_freq
            ),
            "frequency_HIGH": float(
                high_freq
            ),
            "log2_LOW_vs_HIGH": float(
                np.log2(
                    low_freq
                    / high_freq
                )
            ),
        })

    table = pd.DataFrame(
        rows
    )

    kmer_tables.append(
        table
    )

    top_low = (
        table
        .nlargest(
            25,
            "log2_LOW_vs_HIGH",
        )
        .copy()
    )

    top_low[
        "direction"
    ] = "LOW_ENRICHED"

    top_high = (
        table
        .nsmallest(
            25,
            "log2_LOW_vs_HIGH",
        )
        .copy()
    )

    top_high[
        "direction"
    ] = "HIGH_ENRICHED"

    top_kmer_tables.extend(
        [
            top_low,
            top_high,
        ]
    )

kmer_differences = pd.concat(
    kmer_tables,
    ignore_index=True,
)

top_kmers = pd.concat(
    top_kmer_tables,
    ignore_index=True,
)

for stratum_name in STRATA:
    print(
        "\n===",
        stratum_name,
        "LOW-enriched 6-mers ===",
    )

    display(
        top_kmers[
            (
                top_kmers[
                    "stratum"
                ]
                == stratum_name
            )
            & (
                top_kmers[
                    "direction"
                ]
                == "LOW_ENRICHED"
            )
        ][
            [
                "kmer",
                "log2_LOW_vs_HIGH",
                "frequency_LOW",
                "frequency_HIGH",
            ]
        ]
        .head(
            15
        )
    )

    print(
        "===",
        stratum_name,
        "HIGH-enriched 6-mers ===",
    )

    display(
        top_kmers[
            (
                top_kmers[
                    "stratum"
                ]
                == stratum_name
            )
            & (
                top_kmers[
                    "direction"
                ]
                == "HIGH_ENRICHED"
            )
        ][
            [
                "kmer",
                "log2_LOW_vs_HIGH",
                "frequency_LOW",
                "frequency_HIGH",
            ]
        ]
        .head(
            15
        )
    )



=== READS_1 LOW-enriched 6-mers ===


,kmer,log2_LOW_vs_HIGH,frequency_LOW,frequency_HIGH
0,GGGGGG,2.238400,0.000781,0.000166
1,CCCCCC,1.384793,0.000603,0.000231
2,AGGGGG,1.214050,0.000330,0.000142
3,GGGGTG,1.199694,0.000244,0.000106
4,GGGGTA,1.167971,0.000202,0.000090
5,GGGGGC,1.141791,0.000162,0.000073
6,AAAAAA,1.109518,0.002251,0.001043
7,TAGGGG,1.048064,0.000178,0.000086
8,AATAGG,1.036655,0.000280,0.000136
9,TAGGCA,1.035319,0.000193,0.000094


=== READS_1 HIGH-enriched 6-mers ===


,kmer,log2_LOW_vs_HIGH,frequency_LOW,frequency_HIGH
25,GCGGAC,-1.553972,0.000073,0.000216
26,AGGACG,-1.508420,0.000098,0.000279
27,GGACGA,-1.487317,0.000119,0.000333
28,TGGACG,-1.457977,0.000115,0.000317
29,CCGCGG,-1.444655,0.000042,0.000114
30,CGGCGG,-1.414908,0.000049,0.000131
31,ATGGCG,-1.380475,0.000185,0.000483
32,GGTAAG,-1.354220,0.000205,0.000524
33,GATCGC,-1.334290,0.000080,0.000203
34,TGGCGG,-1.314857,0.000146,0.000362



=== READS_2 LOW-enriched 6-mers ===


...

,kmer,log2_LOW_vs_HIGH,frequency_LOW,frequency_HIGH
75,TCGACG,-2.087463,0.000049,0.000206
76,CCGAAG,-1.770518,0.000052,0.000176
77,CACCGA,-1.628031,0.000067,0.000206
78,CGACGG,-1.584963,0.000043,0.000128
79,ACGGAG,-1.584963,0.000076,0.000228
80,TCACCG,-1.563901,0.000070,0.000206
81,GACTCG,-1.545968,0.000076,0.000222
82,CGACGA,-1.537657,0.000094,0.000273
83,ATGGCG,-1.513409,0.000188,0.000537
84,CGGGTC,-1.502500,0.000036,0.000103



=== READS_3_4 LOW-enriched 6-mers ===


,kmer,log2_LOW_vs_HIGH,frequency_LOW,frequency_HIGH
100,CCGTAC,1.523562,0.000103,0.000036
101,CAGGGG,1.392317,0.000188,0.000072
102,CCCATT,1.308122,0.000233,0.000094
103,GGGGGG,1.285402,0.000350,0.000144
104,AACCTA,1.273018,0.000260,0.000108
105,TATATA,1.169925,0.000565,0.000251
106,CCTCTA,1.148863,0.000229,0.000103
107,CTTACA,1.130397,0.000363,0.000166
108,GGACAT,1.111893,0.000300,0.000139
109,TCTATA,1.108524,0.000309,0.000144


=== READS_3_4 HIGH-enriched 6-mers ===


,kmer,log2_LOW_vs_HIGH,frequency_LOW,frequency_HIGH
125,CGGACT,-2.039528,0.000040,0.000166
126,GGCGGA,-2.017922,0.000090,0.000363
127,GCGGAC,-1.974005,0.000063,0.000247
128,CGGAAC,-1.969626,0.000054,0.000211
129,ATGGCG,-1.749192,0.000211,0.000709
130,CGCGGA,-1.710493,0.000049,0.000161
131,GCGGAA,-1.695145,0.000094,0.000305
132,CGGACG,-1.684498,0.000063,0.000202
133,GCGGAT,-1.643856,0.000072,0.000224
134,CGCACT,-1.618910,0.000063,0.000193


In [12]:
# ============================================================
# ONE-SCREEN SUMMARY
# ============================================================

summary_rows = []

for stratum_name in STRATA:
    group_summary = (
        cohort_summary[
            cohort_summary[
                "stratum"
            ]
            == stratum_name
        ]
        .set_index(
            "score_group"
        )
    )

    probes = (
        probe_results[
            probe_results[
                "stratum"
            ]
            == stratum_name
        ]
        .set_index(
            "feature_set"
        )
    )

    pos = (
        position_profile_summary[
            position_profile_summary[
                "stratum"
            ]
            == stratum_name
        ]
        .iloc[
            0
        ]
    )

    high = (
        group_summary.loc[
            "HIGH_SCORE_TP"
        ]
    )

    low = (
        group_summary.loc[
            "LOW_SCORE_TP"
        ]
    )

    summary_rows.append({
        "stratum": stratum_name,
        "n_per_group": int(
            high[
                "n"
            ]
        ),
        "score_high_median": float(
            high[
                "score_median"
            ]
        ),
        "score_low_median": float(
            low[
                "score_median"
            ]
        ),
        "gc_core_high": float(
            high[
                "gc_core32_mean"
            ]
        ),
        "gc_core_low": float(
            low[
                "gc_core32_mean"
            ]
        ),
        "gc_context_high": float(
            high[
                "gc_context256_mean"
            ]
        ),
        "gc_context_low": float(
            low[
                "gc_context256_mean"
            ]
        ),
        "edge_distance_high_median": float(
            high[
                "dominant_tss_edge_distance_median_bp"
            ]
        ),
        "edge_distance_low_median": float(
            low[
                "dominant_tss_edge_distance_median_bp"
            ]
        ),
        "composition_contig_auc": float(
            probes.loc[
                "composition_4_features",
                "contig_heldout_auc",
            ]
        ),
        "bag4_contig_auc": float(
            probes.loc[
                "bag_of_4mer_256bp",
                "contig_heldout_auc",
            ]
        ),
        "position_contig_auc": float(
            probes.loc[
                "position_aware_base_256bp",
                "contig_heldout_auc",
            ]
        ),
        "bag4_random_auc": float(
            probes.loc[
                "bag_of_4mer_256bp",
                "random_split_auc",
            ]
        ),
        "position_random_auc": float(
            probes.loc[
                "position_aware_base_256bp",
                "random_split_auc",
            ]
        ),
        "max_position_frequency_diff_256bp": float(
            pos[
                "max_abs_base_frequency_diff_256bp"
            ]
        ),
    })

final_summary = pd.DataFrame(
    summary_rows
)

display(
    final_summary
)


,stratum,n_per_group,score_high_median,score_low_median,gc_core_high,gc_core_low,gc_context_high,gc_context_low,edge_distance_high_median,edge_distance_low_median,composition_contig_auc,bag4_contig_auc,position_contig_auc,bag4_random_auc,position_random_auc,max_position_frequency_diff_256bp
0,READS_1,3402,0.942000,0.292174,0.464093,0.411532,0.438986,0.403111,8.0,7.0,0.682043,0.748976,0.697598,0.756455,0.712369,0.091417
1,READS_2,1296,0.956032,0.361418,0.459129,0.406829,0.437361,0.404318,7.0,7.0,0.688409,0.729741,0.643870,0.726125,0.662017,0.076389
2,READS_3_4,872,0.968472,0.431429,0.461690,0.417037,0.436367,0.408566,7.0,7.0,0.650932,0.696749,0.583788,0.687008,0.580968,0.086009


# Как читать главный результат

Особенно смотри на `contig_heldout_auc`, потому что это более строгая проверка,
чем случайный train/test split.

Если при **одинаковом experimental strength** получится, например:

```text
bag-of-4mer AUC       ~0.70+
position-aware AUC    ~0.65+
```

то это сильный аргумент, что текущая CNN систематически предпочитает один
sequence-класс настоящих TSS другому.

Если AUC упадут близко к `0.5`, значит большая часть прежнего HIGH/LOW различия
действительно была связана с strength confound.

Также сравни:

```text
dominant_tss_edge_distance_median_bp
```

Если LOW_SCORE_TP заметно ближе к краям 32-bp bins, часть ошибки может быть
следствием coarse localization.


In [13]:
# ============================================================
# FINAL CELL — SAVE *ALL* EXPERIMENT RESULTS INTO ONE JSON FILE
# ============================================================

import json
from datetime import datetime


RESULTS_DIR = (
    RUN_DIR
    / "strength_matched_tp_study"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_JSON = (
    RESULTS_DIR
    / "EXP033_STRENGTH_MATCHED_TP_RESULTS.json"
)


def records_for_json(
    frame,
):
    clean = frame.copy()

    # JSON-safe NaN handling.
    clean = clean.where(
        pd.notnull(
            clean
        ),
        None,
    )

    return clean.to_dict(
        orient="records"
    )


results_payload = {
    "study": {
        "name": "EXP033_STRENGTH_MATCHED_TP_STUDY",
        "source_experiment": "CNN-EXP-033",
        "checkpoint_step": 10000,
        "created_at": datetime.now().isoformat(
            timespec="seconds"
        ),
        "region_width_bp": int(
            REGION_WIDTH
        ),
        "sequence_context_bp": int(
            CONTEXT_BP
        ),
        "tail_fraction": float(
            TAIL_FRACTION
        ),
        "min_match_cell_size": int(
            MIN_CELL_SIZE
        ),
        "matching_columns": list(
            MATCH_COLUMNS
        ),
        "strata": {
            key: list(
                value
            )
            for key, value in STRATA.items()
        },
        "class_definition": {
            "HIGH_SCORE_TP": (
                "top score tail inside each exact "
                "contig/read_sum/tss_count/read_max match cell"
            ),
            "LOW_SCORE_TP": (
                "bottom score tail inside each exact "
                "contig/read_sum/tss_count/read_max match cell"
            ),
        },
    },

    "retention_summary": records_for_json(
        retention_summary
    ),

    "match_cells": records_for_json(
        match_cells
    ),

    "cohort_summary": records_for_json(
        cohort_summary
    ),

    "probe_results": records_for_json(
        probe_results
    ),

    "probe_contig_folds": records_for_json(
        probe_folds
    ),

    "position_profile_summary": records_for_json(
        position_profile_summary
    ),

    # Full position-by-base frequency profiles.
    "position_profile_full": records_for_json(
        position_profile_full
    ),

    # All 4096-ish 6-mer comparisons for each stratum.
    "kmer6_all": records_for_json(
        kmer_differences
    ),

    "kmer6_top": records_for_json(
        top_kmers
    ),

    "final_summary": records_for_json(
        final_summary
    ),
}

RESULTS_JSON.write_text(
    json.dumps(
        results_payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

# Optional detailed CSVs for local inspection.
retention_summary.to_csv(
    RESULTS_DIR
    / "retention_summary.csv",
    index=False,
)

cohort_summary.to_csv(
    RESULTS_DIR
    / "cohort_summary.csv",
    index=False,
)

probe_results.to_csv(
    RESULTS_DIR
    / "probe_results.csv",
    index=False,
)

probe_folds.to_csv(
    RESULTS_DIR
    / "probe_contig_folds.csv",
    index=False,
)

position_profile_summary.to_csv(
    RESULTS_DIR
    / "position_profile_summary.csv",
    index=False,
)

position_profile_full.to_csv(
    RESULTS_DIR
    / "position_profile_full.csv",
    index=False,
)

kmer_differences.to_csv(
    RESULTS_DIR
    / "kmer6_all.csv",
    index=False,
)

top_kmers.to_csv(
    RESULTS_DIR
    / "kmer6_top.csv",
    index=False,
)

final_summary.to_csv(
    RESULTS_DIR
    / "final_summary.csv",
    index=False,
)

# Save selected cohort metadata/sequences separately for reproducibility,
# while the single JSON above contains all experiment RESULT TABLES.
matched.to_csv(
    RESULTS_DIR
    / "matched_regions_with_sequences.csv",
    index=False,
)

print()
print("=" * 70)
print("ГОТОВО")
print("=" * 70)
print()
print("Все результаты экспериментов собраны в ОДИН файл:")
print()
print(RESULTS_JSON)
print()
print(
    "JSON size:",
    f"{RESULTS_JSON.stat().st_size / 1024 / 1024:.2f} MB",
)
print()
print("ЗАГРУЗИ МНЕ ИМЕННО ЭТОТ JSON.")
print()
print("Дополнительные CSV сохранены рядом только для локальной проверки.")



ГОТОВО

Все результаты экспериментов собраны в ОДИН файл:

D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP033_BASELEVEL_RF2053_REGIONAL_32_ONLY\run_001\strength_matched_tp_study\EXP033_STRENGTH_MATCHED_TP_RESULTS.json

JSON size: 3.84 MB

ЗАГРУЗИ МНЕ ИМЕННО ЭТОТ JSON.

Дополнительные CSV сохранены рядом только для локальной проверки.
